# FitMe AI CLEAN MVP v2 — FASHN VTON 1.5

This version uses **async GPU jobs**, so long generation does not sit inside one Cloudflare request.

1. Kaggle → **Settings → Accelerator → GPU**.
2. Internet must be **ON**.
3. Press **Run All**.
4. At the end copy `FITME_AI_URL=https://...trycloudflare.com`.
5. On Windows run `START_FITME.bat` and paste that line.


In [ ]:
!nvidia-smi
%pip install -q "fashn-vton @ git+https://github.com/fashn-AI/fashn-vton-1.5.git" fastapi uvicorn python-multipart httpx
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
print('Dependencies ready')


In [ ]:
from pathlib import Path
server_code = r'''\
from __future__ import annotations

import io
import os
import threading
import time
import uuid

import torch
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import Response
from huggingface_hub import hf_hub_download
from PIL import Image, ImageOps
from fashn_vton import TryOnPipeline

WEIGHTS_DIR = "/kaggle/working/fitme_weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(os.path.join(WEIGHTS_DIR, "dwpose"), exist_ok=True)

print("Downloading/checking FASHN weights...")
hf_hub_download(
    repo_id="fashn-ai/fashn-vton-1.5",
    filename="model.safetensors",
    local_dir=WEIGHTS_DIR,
)
for filename in ("yolox_l.onnx", "dw-ll_ucoco_384.onnx"):
    hf_hub_download(
        repo_id="fashn-ai/DWPose",
        filename=filename,
        local_dir=os.path.join(WEIGHTS_DIR, "dwpose"),
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA/GPU is not enabled in Kaggle. Turn GPU on first.")

if torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("GPU:", torch.cuda.get_device_name(0))
print("Loading FASHN VTON 1.5...")
pipeline = TryOnPipeline(weights_dir=WEIGHTS_DIR, device="cuda")
print("FASHN READY")

app = FastAPI(title="FitMe FASHN Remote", version="1.1-async")
pipeline_lock = threading.Lock()
jobs_lock = threading.Lock()
JOBS = {}
RESULTS = {}

CATEGORY_MAP = {
    "upper_body": "tops",
    "lower_body": "bottoms",
    "dresses": "one-pieces",
    "tops": "tops",
    "bottoms": "bottoms",
    "one-pieces": "one-pieces",
}


@app.get("/health")
def health():
    return {
        "status": "ready",
        "engine": "FASHN VTON 1.5",
        "mode": "async-jobs",
        "gpu": torch.cuda.get_device_name(0),
    }


def generate_job(
    job_id: str,
    person_bytes: bytes,
    garment_bytes: bytes,
    category: str,
    garment_photo_type: str,
    num_timesteps: int,
    guidance_scale: float,
    seed: int,
    segmentation_free: bool,
):
    try:
        with jobs_lock:
            JOBS[job_id]["status"] = "processing"
            JOBS[job_id]["started_at"] = time.time()

        person = Image.open(io.BytesIO(person_bytes))
        person = ImageOps.exif_transpose(person).convert("RGB")
        garment = Image.open(io.BytesIO(garment_bytes))
        garment = ImageOps.exif_transpose(garment).convert("RGB")

        mapped = CATEGORY_MAP.get(category, "tops")
        photo_type = (
            garment_photo_type
            if garment_photo_type in {"flat-lay", "model"}
            else "flat-lay"
        )
        steps = max(10, min(int(num_timesteps), 50))
        cfg = max(1.0, min(float(guidance_scale), 3.0))

        with pipeline_lock:
            result = pipeline(
                person_image=person,
                garment_image=garment,
                category=mapped,
                garment_photo_type=photo_type,
                num_samples=1,
                num_timesteps=steps,
                guidance_scale=cfg,
                seed=int(seed),
                segmentation_free=bool(segmentation_free),
            )

        output = io.BytesIO()
        result.images[0].save(output, format="PNG")

        with jobs_lock:
            RESULTS[job_id] = output.getvalue()
            JOBS[job_id]["status"] = "completed"
            JOBS[job_id]["completed_at"] = time.time()

    except Exception as exc:
        with jobs_lock:
            JOBS[job_id]["status"] = "failed"
            JOBS[job_id]["error"] = f"{type(exc).__name__}: {str(exc)[:1000]}"


@app.post("/jobs")
async def create_job(
    person_photo: UploadFile = File(...),
    garment_photo: UploadFile = File(...),
    category: str = Form("tops"),
    garment_photo_type: str = Form("flat-lay"),
    num_timesteps: int = Form(20),
    guidance_scale: float = Form(1.5),
    seed: int = Form(42),
    segmentation_free: bool = Form(True),
):
    person_bytes = await person_photo.read()
    garment_bytes = await garment_photo.read()

    if not person_bytes or not garment_bytes:
        raise HTTPException(status_code=400, detail="Empty input image")

    job_id = uuid.uuid4().hex[:12]
    with jobs_lock:
        JOBS[job_id] = {
            "job_id": job_id,
            "status": "queued",
            "created_at": time.time(),
            "error": None,
        }

    threading.Thread(
        target=generate_job,
        args=(
            job_id,
            person_bytes,
            garment_bytes,
            category,
            garment_photo_type,
            num_timesteps,
            guidance_scale,
            seed,
            segmentation_free,
        ),
        daemon=True,
        name=f"fashn-{job_id}",
    ).start()

    return {"job_id": job_id, "status": "queued"}


@app.get("/jobs/{job_id}")
def job_status(job_id: str):
    with jobs_lock:
        job = JOBS.get(job_id)
        if job is None:
            raise HTTPException(status_code=404, detail="Job not found")
        return dict(job)


@app.get("/jobs/{job_id}/result")
def job_result(job_id: str):
    with jobs_lock:
        job = JOBS.get(job_id)
        if job is None:
            raise HTTPException(status_code=404, detail="Job not found")
        if job.get("status") != "completed":
            raise HTTPException(status_code=409, detail=job.get("status"))
        data = RESULTS.get(job_id)

    if not data:
        raise HTTPException(status_code=404, detail="Result not found")

    return Response(content=data, media_type="image/png")


# Short sync route kept only for quick manual tests.
@app.post("/tryon")
async def tryon_sync_disabled():
    raise HTTPException(
        status_code=410,
        detail="Use async /jobs endpoint in CLEAN MVP v2",
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=7860)
'''
Path('/kaggle/working/fitme_remote_server.py').write_text(server_code, encoding='utf-8')
print('Async FASHN server file created')


In [ ]:
import os, re, subprocess, time, requests
from pathlib import Path

for p in ['fitme_server_proc', 'fitme_tunnel_proc']:
    old = globals().get(p)
    if old is not None:
        try: old.terminate()
        except: pass

server_log = open('/kaggle/working/fitme_server.log', 'w')
fitme_server_proc = subprocess.Popen(
    ['python', '/kaggle/working/fitme_remote_server.py'],
    stdout=server_log, stderr=subprocess.STDOUT,
)

print('Waiting for FASHN model/server...')
ready = False
for i in range(180):
    time.sleep(2)
    try:
        r = requests.get('http://127.0.0.1:7860/health', timeout=2)
        if r.status_code == 200:
            print('LOCAL HEALTH:', r.json())
            ready = True
            break
    except Exception:
        pass
    if i and i % 15 == 0:
        print(f'...still loading ({i*2}s)')

if not ready:
    print(Path('/kaggle/working/fitme_server.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('FASHN server did not become ready')

fitme_tunnel_proc = subprocess.Popen(
    ['/kaggle/working/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7860', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

public_url = None
deadline = time.time() + 60
while time.time() < deadline:
    line = fitme_tunnel_proc.stdout.readline()
    if line:
        print(line.rstrip())
        m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
        if m:
            public_url = m.group(0)
            break

if not public_url:
    raise RuntimeError('Cloudflare tunnel URL was not found')

print('\n' + '='*70)
print('FITME_AI_URL=' + public_url)
print('='*70)
print('Leave this Kaggle session running while using FitMe.')
